# 07 — Freeze, evaluate on VAL, then ONCE on TEST

**Purpose.** Freeze the configuration (the three runs and their checkpoint hashes), evaluate Model B (gloss), Model B (text), B0 and B1 and Model A on **VAL** with bootstrap 95% intervals and error analysis, then run the single **final TEST** evaluation, which writes `outputs/FINAL_TEST_LOCK.json`.

**Kaggle settings.** Accelerator **None or GPU** (evaluation is light) · Internet **On** (sacrebleu).

**Inputs.** Dataset `bdsl-k400` + **Add Input → Output of notebook 06**.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `outputs/frozen_config.json`, `outputs/eval/val_<time>/`, `outputs/eval/FINAL_test_<time>/`, `outputs/predictions/`, `outputs/FINAL_TEST_LOCK.json`.

**Expected run time** [ESTIMATE, not measured yet]. 10–20 minutes.

**What to check.** The frozen runs are the ones from 04/05/06 (no NFP_, no DRYRUN_, no overfit). The final cell runs once and prints `lock_written: true`. **Run this notebook only once** (see the warning below).

> ⚠️ **Never re-run this notebook after a successful version.** A new version starts without the lock unless you add this notebook's Output as Input; then a second test run is refused. Re-running the test means the test is no longer clean.
> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "07_baselines_eval"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

In [ ]:
S.pip_install("sacrebleu==2.6.0")
rc_list = S.bdsl("list-runs")

## Step 1 — Freeze (newest finished, for-paper run of each kind)

In [ ]:
rc_freeze = S.bdsl("freeze")
frozen = json.load(open(S.state_root / "outputs/frozen_config.json"))
print(json.dumps({r: v["run_id"] for r, v in frozen["runs"].items()}, indent=1))

## Step 2 — VAL evaluation (bootstrap 95% CI, paired differences, error analysis)

In [ ]:
rc_val = S.bdsl("evaluate", "--split", "val")

## Step 3 — The single FINAL TEST evaluation
The configuration is frozen above. This cell reads the test split for the first and only time.

In [ ]:
rc_final = S.bdsl("evaluate", "--final")
lock = json.load(open(S.state_root / "outputs/FINAL_TEST_LOCK.json"))
print("lock:", lock["created_utc"], lock["results"])

## Step 4 — Test results (measured; small-scale, signer-dependent)

In [ ]:
import pandas as pd
from IPython.display import display
m = json.load(open(S.state_root / "outputs/eval" / lock["results"] / "metrics.json"))
b = m["model_b_and_baselines"]["methods"]
KEYS = ("mpjpe_body", "dtw_body", "mpjpe_lhand", "mpjpe_rhand")
rows = []
for k, v in b.items():
    vals = {x: v["mean_sd_over_seeds"][x]["mean"] for x in KEYS} if "mean_sd_over_seeds" in v else {x: v["summary"][x] for x in KEYS}
    rows.append({"method": k, **{x: round(y, 3) for x, y in vals.items()}})
display(pd.DataFrame(rows))
for comp, d in m["model_b_and_baselines"]["paired_differences"].items():
    x = d["dtw_body"]
    print(f"{comp:30s} DTW body diff {x['estimate']:+.3f} [{x['low']:+.3f}, {x['high']:+.3f}] excludes 0: {x['excludes_zero']}")
print("Model A (agreement with the rule glosser, G0 pseudo-gloss):", {k: round(v["mean"], 1) for k, v in m["model_a"]["mean_sd_over_seeds"].items()})

In [ ]:
S.finish(checks=[
    ("configuration frozen before the test", rc_freeze == 0),
    ("VAL evaluation done", rc_val == 0),
    ("final TEST evaluation done once, lock written", rc_final == 0 and not lock.get("history")),
])